In [ ]:
import camelot

import numpy as np
import pandas as pd

import re

import os

In [ ]:
folder = "../data/barclays_statements/"
file = "2024-05-03 Statement 03-MAY-24 AC 43021297 05073028.pdf"
file_path = folder + file
year = file.split("-")[0]

In [ ]:
table_list= camelot.read_pdf(file_path, pages="2,3", flavor='network')
table1 = table_list[1].df.loc[4:, :]
table2 = table_list[2].df.loc[3:, :]

print(file, table1.shape[1], table2.shape[1])

In [ ]:
table1

In [ ]:
# combine tables
df = pd.concat([table1, table2])

# rename columns
df.columns = ["Date", "Description", "Money out", "Money in", "Balance"]

# reset index
df = df.reset_index(drop=True)

# combine descriptions with multiple rows
prev_not_empty = 0
for id, row in df.iterrows():
    if row.loc["Date"] == "" and row.loc["Money out"] == "" and row.loc["Money in"] == "" and row.loc["Balance"] == "":
        df.loc[prev_not_empty, "Description"] = df.loc[prev_not_empty, "Description"] + " " + row.loc["Description"]
        df.loc[id, "Description"] = ""
    else:
        prev_not_empty = id

# drop empty rows
df.replace('', np.nan, inplace=True).dropna(how='all', inplace=True)

# split start balance into description
df.loc[0, ["Date", "Description"]] = df.loc[0, "Date"].split("  ")
df

# process dates
prev_not_empty = 0
for id, row in df.iterrows():
    if pd.isna(row.loc["Date"]):
        df.loc[id, "Date"] = df.loc[prev_not_empty, "Date"]
    else:
        prev_not_empty = id
df.loc[:, "Date"] =  df.loc[:, "Date"] + " " + year
df

In [ ]:
df.loc[:, "Date_pd"] =  pd.to_datetime(df.loc[:, "Date"], format="%d %b %Y")
df

In [ ]:
folder = "../data/barclays_statements/"
for file in os.listdir(folder):
    file_path = folder + file

    table_list= camelot.read_pdf(file_path, pages="2,3", flavor='network')
    table1 = table_list[1].df.loc[4:, :]
    table2 = table_list[2].df.loc[3:, :]

    print(file, table1.shape[1], table2.shape[1])